In [ ]:
!pip3 install open_clip_torch transformers scikit-multilearn
!pip3 install trimap
!pip3 install umap-learn
!pip3 install captum
!pip3 install torchcam
!pip3 install grad-cam

In [ ]:
import open_clip
from open_clip import create_model_from_pretrained, get_tokenizer # works on open-clip-torch>=2.23.0, timm>=0.9.8


def initializeModel():
    
    biomedclip_model, biomedclip_preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
    biomedclip_tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

    return biomedclip_model, biomedclip_preprocess,biomedclip_tokenizer

biomedclip_model, biomedclip_preprocess,biomedclip_tokenizer =initializeModel()

### REUSABLE FUNC

In [ ]:
import torch
import os
import pandas as pd
import numpy as np
import pandas as pd
import tqdm
from PIL import Image
from pathlib import Path

import random
from typing import List
from collections import OrderedDict
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, silhouette_score,
                             davies_bouldin_score)
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, accuracy_score,
                             label_ranking_average_precision_score as lrap,
                             coverage_error, label_ranking_loss)
from torchvision.utils import make_grid
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from torch.nn.functional import cosine_similarity, softmax

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import gridspec

# XAI libs
from captum.attr import LayerGradCam
from torchcam.methods import GradCAMpp
from torchcam.utils import overlay_mask
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


#### FUNCTIONS FOR METRIC

In [ ]:

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

classes = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices", "No Finding"
]

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()


def expand_multilabel_fixed(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure every df has *all* 14 binary columns; assumes df.target is list."""
    if 'target' in df.columns:
        for lbl in classes:
            df[lbl] = df['target'].apply(lambda ls: 1 if lbl in ls else 0)
    for lbl in classes:
        if lbl not in df.columns:
            df[lbl] = 0
    return df
def extract_image_embeddings(image_paths: List[str], preprocess, model, pretrained: bool):
    if pretrained:
        model.load_state_dict(torch.load('re_biomedclip_model_paramsb32_e30_lr1e6_optAdam_temp_017_cosineSchedular.pth'))
    model.to(device).eval()
    embeds = []
    for p in image_paths:
        img = preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
        with torch.no_grad():
            e = model.encode_image(img)
        embeds.append(e.cpu())
    return torch.cat(embeds)

def extract_text_embeddings(text_prompts: List[str], tokenizer, model, pretrained: bool):
    if pretrained:
        model.load_state_dict(torch.load('re_biomedclip_model_paramsb32_e30_lr1e6_optAdam_temp_017_cosineSchedular.pth'))
    model.to(device).eval()
    tokens = tokenizer(text_prompts).to(device)
    with torch.no_grad():
        text_emb = model.encode_text(tokens)
    return text_emb.cpu()


def cosine_logits(img_emb: torch.Tensor, txt_emb: torch.Tensor, logit_scale: float = None):
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
    txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
    logits = img_emb @ txt_emb.T
    if logit_scale is not None:
        logits *= logit_scale
    return logits

def validate_with_embeddings(model, df_test: pd.DataFrame, preprocess, tokenizer,
                              classes: list[str], pretrained_flag: bool):
    """Compute logits from pre‑extracted image/text embeddings."""
    img_paths = df_test.filename.tolist()
    img_emb = extract_image_embeddings(img_paths, preprocess, model, pretrained_flag)
    txt_emb = extract_text_embeddings(classes, tokenizer, model, pretrained_flag)
    log_scale = model.logit_scale.exp().cpu() if hasattr(model, "logit_scale") else torch.tensor(1.0)
    logits = (img_emb @ txt_emb.T) * log_scale
    y_true = df_test[classes].values.astype(int)
    return evaluate_classification(logits.numpy(), y_true, classes)

def evaluate_classification(logit, y_true, classes):
    y_pred = (logit > 0).astype(int)
    rep = classification_report(y_true, y_pred, target_names=classes, zero_division=0, output_dict=True)
    return {
        "macro_f1": rep["macro avg"]["f1-score"],
        "per_class_f1": [rep[c]["f1-score"] for c in classes],
        "auroc": roc_auc_score(y_true, logit, average=None).tolist(),
        "auprc": average_precision_score(y_true, logit, average=None).tolist(),
        "mcc": [matthews_corrcoef(y_true[:, i], y_pred[:, i]) for i in range(len(classes))],
    }

def _row_minmax(x: np.ndarray) -> np.ndarray:
    """Scale each row to [0,1] so relative scores matter."""
    mn = x.min(1, keepdims=True); mx = x.max(1, keepdims=True)
    return (x - mn) / (mx - mn + 1e-8)

from scipy.special import expit

def logits_to_prob(logits: np.ndarray, mode: str = "cosine", tau: float = 80.0):
    if mode == "cosine":
        # cosine on CPU numpy
        norm = np.linalg.norm(logits, axis=1, keepdims=True) + 1e-8
        probs = (logits / norm)          # already cos because txt_emb unit-norm
        return (probs + 1) / 2
    elif mode == "sigmoid":
        return expit(logits / tau)
    elif mode == "minmax":
        mn = logits.min(1, keepdims=True); mx = logits.max(1, keepdims=True)
        return (logits - mn) / (mx - mn + 1e-8)
    else:
        raise ValueError("mode must be cosine | sigmoid | minmax")


def compute_metrics(logits: np.ndarray, y_true: np.ndarray):
    y_prob = 1/(1+np.exp(-logits))           # sigmoid
    y_pred = (y_prob > 0.5).astype(int)
    macro_f1 = classification_report(y_true, y_pred, zero_division=0,
                                     output_dict=True)['macro avg']['f1-score']
    auroc    = roc_auc_score(y_true, y_prob, average=None)
    auprc    = average_precision_score(y_true, y_prob, average=None)
    exact_match = (y_pred == y_true).all(1).mean()
    per_class_acc = (y_pred == y_true).mean(0)
    return macro_f1, auroc, auprc, exact_match, per_class_acc


def compute_metrics_v2(logits: np.ndarray, y_true: np.ndarray,threshold:float):
    """Return dict with classification + ranking metrics."""
    y_prob = _row_minmax(logits)          # [0,1] per‑row scaling
    y_pred = (y_prob > threshold).astype(int)
  

    macro_f1 = classification_report(
        y_true, y_pred, zero_division=0, output_dict=True
    )["macro avg"]["f1-score"]
    auroc  = roc_auc_score(y_true, y_prob, average=None).tolist()
    auprc  = average_precision_score(y_true, y_prob, average=None).tolist()

    exact_match = accuracy_score(y_true, y_pred)
    lrap_score  = lrap(y_true, y_prob)
    cov_err     = coverage_error(y_true, y_prob)
    rank_loss   = label_ranking_loss(y_true, y_prob)

    return {
        "macro_f1": macro_f1,
        "auroc": auroc,
        "auprc": auprc,
        "exact_match": exact_match,
        "lrap": lrap_score,
        "coverage_error": cov_err,
        "rank_loss": rank_loss,
    }

from typing import List, Dict
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics_v3(
        logits: np.ndarray,
        y_true: np.ndarray,
        threshold: float,
        class_names: List[str]
) -> Dict[str, Dict[str, float]]:
    """
    Per-disease precision / recall / F1 at a fixed threshold.

    * `logits`  : raw model scores, shape (N, C)
    * `y_true`  : binary ground truth, shape (N, C)
    * `threshold`: decision cut-off applied after row-min-max scaling
    * `class_names`: list of length C (same order as logits columns)

    Returns
    -------
    {class_name: {'precision': p, 'recall': r, 'f1': f}}
    """

    # 1) Convert logits → 0-1 via row-min-max, then threshold
    y_prob = _row_minmax(logits)                       # (N, C)
    y_pred = (y_prob > threshold).astype(int)

    # 2) Compute per-class metrics
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # 3) Build readable dict
    per_class = {
        name: {
            "precision": float(p),
            "recall":    float(r),
            "f1":        float(f)
        }
        for name, p, r, f in zip(class_names, prec, rec, f1)
    }
    return per_class

import numpy as np
from typing import List, Dict
from scipy.special import expit        # fast sigmoid
from sklearn.metrics import (
    precision_recall_fscore_support,
)

def compute_metrics_v3_sigmoid(
        logits: np.ndarray,            # (N, C) raw similarity scores
        y_true: np.ndarray,            # (N, C) binary ground-truth
        threshold: float,
        class_names: List[str]
) -> Dict[str, Dict[str, float]]:
    """
    Per-class precision / recall / F1 at a fixed sigmoid threshold.

    Returns {class_name: {'precision': p, 'recall': r, 'f1': f}}
    """

    # 1) Convert logits -> probabilities with sigmoid
    y_prob = expit(logits)                            # (N, C) in [0,1]

    # 2) Apply global threshold
    y_pred = (y_prob > threshold).astype(int)

    # 3) Per-class P/R/F1
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    return {
        name: {"precision": float(p), "recall": float(r), "f1": float(f)}
        for name, p, r, f in zip(class_names, prec, rec, f1)
    }


def compute_metrics_v4(logits, y_true, threshold, class_names,
                       prob_mode="cosine", tau=80.0):
    y_prob = logits_to_prob(logits, mode=prob_mode, tau=tau)
    y_pred = (y_prob > threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0)
    return {
        cls: {"precision": float(p), "recall": float(r), "f1": float(f)}
        for cls, p, r, f in zip(class_names, prec, rec, f1)
    }






def compute_topk_accuracy(y_true: np.ndarray, logits: np.ndarray, ks=(1,2,3)):
    """Return a mapping  {disease: [top1, top2, top3]} (per‑class top‑k recall).

    A hit occurs when **any** ground‑truth‑positive image ranks that class
    inside the top‑k of its own logit vector.
    """
    n, C = logits.shape
    ranks = (-logits).argsort(1).argsort(1)          # rank 0 = largest score
    y_true = y_true.astype(bool)

    # scores[k][c] = hits / positives for class c
    acc_mat = {k: np.zeros(C, dtype=float) for k in ks}
    pos = y_true.sum(0).clip(min=1)                  # positives per class
    print(pos)
    for k in ks:
        hits = ((ranks < k) & y_true).sum(0)
        acc_mat[k] = hits / pos

    # build disease → [top1, top2, top3]
    out = {classes[c]: [acc_mat[k][c] for k in ks] for c in range(C)}
    return out


def compute_separability(embeddings: np.ndarray, labels: np.ndarray):
    y_flat = labels.argmax(1)
    # Logistic probe
    logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
    logreg.fit(embeddings, y_flat)
    probe_acc = logreg.score(embeddings, y_flat)
    # kNN
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine'); knn.fit(embeddings, y_flat)
    knn_acc = knn.score(embeddings, y_flat)
    # Cluster metrics
    sil = silhouette_score(embeddings, y_flat, metric='cosine')
    db  = davies_bouldin_score(embeddings, y_flat)
    # Fisher ratio
    means = [embeddings[y_flat==c].mean(0) for c in np.unique(y_flat)]
    overall_mean = embeddings.mean(0)
    sb = sum(len(embeddings[y_flat==c]) * np.sum((m-overall_mean)**2) for c, m in enumerate(means))
    sw = sum(((embeddings[y_flat==c]-means[c])**2).sum() for c in range(len(means)))
    fisher = sb / (sw + 1e-9)
    # Inter / Intra distances
    intra_d = np.mean([np.linalg.norm(embeddings[y_flat==c] - means[c], axis=1).mean() for c in np.unique(y_flat)])
    centroid_pairs = [np.linalg.norm(means[i]-means[j]) for i in range(len(means)) for j in range(i+1,len(means))]
    inter_d = float(np.mean(centroid_pairs))
    ic_ia = inter_d / (intra_d + 1e-9)
    return probe_acc, knn_acc, sil, db, fisher, intra_d, inter_d, ic_ia

def select_samples(test_df: pd.DataFrame, logits: np.ndarray, n_classes=5):
    """Pick n_classes distinct labels each with one TP and one FN."""
    y_true = test_df[classes].values.astype(int)
    y_pred = (logits > 0).astype(int)
    corr_paths, wrong_paths, labels = [], [], []
    print(y_true,y_pred)
    for c in np.random.permutation(len(classes)):
        # True positive
        tp_idx = np.where((y_true[:,c]==1) & (y_pred[:,c]==1))[0]
        fn_idx = np.where((y_true[:,c]==1) & (y_pred[:,c]==0))[0]
        if len(tp_idx) and len(fn_idx):
            i_corr = int(random.choice(tp_idx))
            i_wrong= int(random.choice(fn_idx))
            corr_paths.append(test_df.iloc[i_corr]['filename'])
            wrong_paths.append(test_df.iloc[i_wrong]['filename'])
            labels.append(classes[c])
            if len(labels)==n_classes:
                break
    return corr_paths, wrong_paths, labels


#### FUNCTIONS FOR VISUALIZATIONS

In [ ]:
def returnVitLayer(model):
    return model.visual.to(device).eval()
vit = returnVitLayer(biomedclip_model)

# ─────────────────────  2. TARGET LAYER FOR GRAD-CAM  ──────────────────
# Use the projection of the last attention block

def targetlayer(N):
    return vit.trunk.blocks[-N].attn.proj
target_layer = targetlayer(3)

# ─────────────────────  3. CUSTOM SCALAR TARGET  ───────────────────────
class EmbeddingNormTarget:
    """Return ‖embedding‖₂ so Grad-CAM has a scalar loss."""
    def __call__(self, model_output):
        return model_output.norm()


def vit_reshape(x, h=14, w=14):
    """
    Convert (B, N, C) ViT tokens to (B, C, H, W).
    CLS token is dropped, remainder reshaped to grid.
    """
    B, N, C = x.shape            # N = 1 + h*w
    x = x[:, 1:, :].permute(0, 2, 1)              # (B, C, h*w)
    return x.reshape(B, C, h, w)

class SimilarityTarget:
    """Scalar = similarity(image emb, prompt[cls_idx])."""
    def __init__(self, cls_idx: int, txt_mat: torch.Tensor):
        self.cls = cls_idx                  # int 0-13
        self.txt = txt_mat                  # (14, D) tensor on same device

    def __call__(self, img_emb: torch.Tensor):
        # img_emb is (D,) or (1, D) depending on torch-cam version
        if img_emb.dim() == 1:              # (D,)
            return torch.dot(img_emb, self.txt[self.cls])
        else:                               # (1, D)
            return (img_emb @ self.txt.T)[0, self.cls]

## ─── 5. Grad-CAM wrapper that handles tensor output ─────────────────────
class GradCAMCustom(GradCAM):
    def forward(self, x, targets, eigen_smooth=False):
        out  = self.model(x)                      # img_emb (B, D)
        loss = sum(t(out) for t in targets)
        loss.backward(retain_graph=True)
        return super().forward(x, targets, eigen_smooth)

cam = GradCAMCustom(
        model=vit,
        target_layers=[target_layer],
        reshape_transform=vit_reshape
)
# ─── 6. Pre-processing / plotting helpers ───────────────────────────────
def preprocess_for_cam(path):
    img = Image.open(path).convert("RGB")
    original = np.asarray(img.resize((224, 224))).astype(np.float32) / 255.0
    tensor   = biomedclip_preprocess(img).unsqueeze(0).to(device)
    return original, tensor

def generate_gradcam(path,cls_idx,txt_emb_tensor):
    orig, inp = preprocess_for_cam(path)
    target_fn = SimilarityTarget(cls_idx, txt_emb_tensor)
    cam_map   = cam(inp, targets=[target_fn])[0]
#     cam_map   = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    if hasattr(cam, "remove_hooks"):
        cam.remove_hooks()
    elif hasattr(cam, "clear_hooks"):
        cam.clear_hooks()
    return orig, show_cam_on_image(orig, cam_map, use_rgb=True)

def plot_gradcam(path,disease,cls_idx,txt_emb_tensor,disease_prob):
    orig, cam_img = generate_gradcam(path,cls_idx,txt_emb_tensor)
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(orig);    ax[0].axis("off"); ax[0].set_title("Original")
    ax[1].imshow(cam_img); ax[1].axis("off"); ax[1].set_title(f"{disease}, Probability: {disease_prob:.3g}",fontsize=18)
    plt.tight_layout(); plt.show()
    if hasattr(cam, "remove_hooks"):
        cam.remove_hooks()
    elif hasattr(cam, "clear_hooks"):
        cam.clear_hooks()



### IUXRAY

In [ ]:

iuxray_df=pd.read_csv('Dataset/iuxray/iuxray_frontal_14labels.csv')

In [ ]:
# List of disease names
disease_names = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", 
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", 
    "Atelectasis", "Pneumothorax", "Pleural Effusion", 
    "Pleural Other", "Fracture", "Support Devices", "No Finding"
]

# Step 1: Aggregate text columns into a single 'report' column
text_columns = ['indication', 'comparison', 'findings', 'impression']
iuxray_df['report'] = iuxray_df[text_columns].fillna('').apply(lambda row: ' '.join(row).strip(), axis=1)

# Step 2: Create a 'target' column by aggregating disease labels where the value is 1
def aggregate_diseases(row):
    return [disease for disease in disease_names if row[disease] == 1]

iuxray_df['target'] = iuxray_df.apply(aggregate_diseases, axis=1)

# Step 3: Select and reorder relevant columns for further processing
iuxray_df = iuxray_df[['uid', 'filename', 'report', 'target']]
# Replace empty lists in the 'target' column with ['No Finding']
iuxray_df['target'] = iuxray_df['target'].apply(
    lambda x: ['No Finding'] if len(x) == 0 else x
)

#### 70-15-15 Split

In [ ]:

uids = iuxray_df["uid"].unique()
print(f"Unique uids in frontal-only set: {len(uids)}")   # 3 818  (one per image)

# 2) reproducible shuffling & three-way split
#    first carve out 30 % for val+test, then split that in half
train_uids, tmp_uids = train_test_split(
    uids, test_size=0.30, random_state=42, shuffle=True
)
val_uids,  test_uids = train_test_split(
    tmp_uids, test_size=0.50, random_state=42, shuffle=True
)

# 3) build the actual DataFrames
train_df_iuxray = iuxray_df[iuxray_df["uid"].isin(train_uids)].reset_index(drop=True)
val_df_iuxray   = iuxray_df[iuxray_df["uid"].isin(val_uids)].reset_index(drop=True)
test_df_iuxray = iuxray_df[iuxray_df["uid"].isin(test_uids)].reset_index(drop=True)

print(f"Train: {len(train_df_iuxray)} images")   # 2 673
print(f"Val  : {len(val_df_iuxray)} images")     #   573
print(f"Test : {len(test_df_iuxray)} images")    #   572


In [ ]:
for disease in disease_names:
    print('#datapoints for disease {}: {}'.format(disease, len(train_df_iuxray[train_df_iuxray['target'].apply(lambda x: disease in x)])))

In [ ]:
for disease in disease_names:
    print('#datapoints for disease {}: {}'.format(disease, len(iuxray_df[iuxray_df['target'].apply(lambda x: disease in x)])))

In [ ]:
import matplotlib.pyplot as plt

# Data
counts = {
    "No Finding": 2400,
    "Cardiomegaly": 415,
    "Lung Opacity": 516,
    "Atelectasis": 216,
    "Support Devices": 163,
    "Lung Lesion": 165,
    "Fracture": 127,
    "Pleural Effusion": 129,
    "Enlarged Cardiomediastinum": 70,
    "Pneumonia": 68,
    "Pleural Other": 56,
    "Edema": 49,
    "Consolidation": 31,
    "Pneumothorax": 20
}

# Sort descending
sorted_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
labels, values = zip(*sorted_items)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, values, color='grey')

# Add count labels
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2, 
        height, 
        f'{height}', 
        ha='center', 
        va='bottom',
        fontsize=9
    )

# Formatting
ax.set_ylabel("Sample Count")
ax.set_title("Number of Samples per Disease")
ax.set_xticklabels(labels, rotation=45, ha='right')
plt.tight_layout(pad=0)
# plt.show()
plt.savefig('data_dist.png')


#### Investigating Zero-shot

In [ ]:
image_paths = test_df_iuxray['filename'].tolist()
image_dir = "Dataset/iuxray/images/images_normalized"  # Path to the directory containing images

# # Extract embeddings
# image_embeddings = extract_image_embeddings(image_paths, biomedclip_preprocess, biomedclip_model, pretrained=False)
# text_embeddings = extract_text_embeddings(text_prompts, biomedclip_tokenizer, biomedclip_model, pretrained=False)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import umap.umap_ as umap
import trimap.trimap_ as trimap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import torch

# Ensure test_df and image_embeddings are reloaded if execution state was reset
# Assuming test_df and image_embeddings are available

# Define "No Finding" vs "Abnormal" categories
binary_categories = ["No Finding", "Abnormal"]

# Define 4 abnormal disease domains
domain_categories = {
    "Cardiovascular": ["Enlarged Cardiomediastinum", "Cardiomegaly"],
    "Pulmonary": ["Pneumonia", "Consolidation", "Atelectasis", "Pneumothorax", 
                  "Pleural Other", "Pleural Effusion", "Edema", "Lung Opacity", "Lung Lesion"],
    "Skeletal": ["Fracture"],
    "Device": ["Support Devices"]
}
test_df=test_df_iuxray.copy()
# List of all 14 original categories
original_categories = list(set(sum(domain_categories.values(), [])) | {"No Finding"})

# Ensure target column is in list format
test_df["target"] = test_df["target"].apply(lambda x: x if isinstance(x, list) else [x])

# Dictionary to store indices for each category
category_indices = {category: test_df[test_df["target"].apply(lambda x: category in x)].index.tolist() for category in original_categories}

# Limit samples per category to prevent memory issues
N = 100  # Adjust as needed
for category in original_categories:
    category_indices[category] = category_indices[category][:N]

# Flatten all indices into a single list
selected_indices = [idx for cat_indices in category_indices.values() for idx in cat_indices]

# Select corresponding image embeddings
selected_embeddings = image_embeddings[selected_indices].cpu().numpy()  # Ensure NumPy format

# Create labels (assign a number for each category)
label_mapping = {category: idx for idx, category in enumerate(original_categories)}
all_labels = [label_mapping[test_df.loc[idx, "target"][0]] for idx in selected_indices]  # Convert category names to numeric labels

# Create binary labels: 0 for "No Finding", 1 for "Abnormal"
binary_labels = [0 if test_df.loc[idx, "target"][0] == "No Finding" else 1 for idx in selected_indices]

# Create domain labels for abnormal diseases
# Assign domain labels for abnormal diseases
domain_labels = []
for idx in selected_indices:
    original_label = test_df.loc[idx, "target"][0]
    if original_label in domain_categories["Cardiovascular"]:
        domain_labels.append(0)  # Cardiovascular
    elif original_label in domain_categories["Pulmonary"]:
        domain_labels.append(1)  # Pulmonary
    elif original_label in domain_categories["Skeletal"]:
        domain_labels.append(2)  # Skeletal
    elif original_label in domain_categories["Device"]:
        domain_labels.append(3)  # Device
    else:
        domain_labels.append(-1)  # Set -1 for "Other" instead of None


# Apply PCA
pca = PCA(n_components=2)
pca_results = pca.fit_transform(selected_embeddings)

# Apply UMAP with fixed random_state for reproducibility
umap_reducer = umap.UMAP(n_components=2, random_state=42)
umap_results = umap_reducer.fit_transform(selected_embeddings)

# Apply TriMap with fixed random_state for reproducibility
trimap_reducer = trimap.TRIMAP(n_dims=2, n_random=42)
trimap_results = trimap_reducer.fit_transform(selected_embeddings)

# Normalize projections using StandardScaler
scaler = StandardScaler()
pca_results_scaled = scaler.fit_transform(pca_results)
umap_results_scaled = scaler.transform(umap_results)
trimap_results_scaled = scaler.transform(trimap_results)

# Assign colors for all categories (first row)
cmap_all = plt.cm.get_cmap("tab20", len(original_categories))
all_colors = [cmap_all(label / len(original_categories)) for label in all_labels]

# Assign colors for "No Finding" vs "Abnormal" (second row)
color_map_binary = {0: "green", 1: "red"}
binary_colors = [color_map_binary[label] for label in binary_labels]

# Convert domain_labels to a NumPy array for filtering
domain_labels = np.array(domain_labels)

# Find valid indices where domain labels are NOT -1
valid_indices = domain_labels != -1

# Filter embeddings to match valid domain labels
filtered_pca_results = pca_results_scaled[valid_indices]
filtered_umap_results = umap_results_scaled[valid_indices]
filtered_trimap_results = trimap_results_scaled[valid_indices]
filtered_domain_labels = domain_labels[valid_indices]  # Now matches size

# Assign colors for domain categories
cmap_domain = plt.cm.get_cmap("tab10", 4)  # 4 categories
domain_colors = [cmap_domain(label) for label in filtered_domain_labels]

# Create 3x3 subplot for PCA, UMAP, and TriMap
fig, axes = plt.subplots(3, 3, figsize=(18, 18))

# First row: All 14 categories
axes[0, 0].scatter(pca_results_scaled[:, 0], pca_results_scaled[:, 1], c=all_colors, s=80, alpha=0.8)
axes[0, 0].set_title("PCA - All Labels")
axes[0, 0].set_xlabel("PCA Component 1")
axes[0, 0].set_ylabel("PCA Component 2")

axes[0, 1].scatter(umap_results_scaled[:, 0], umap_results_scaled[:, 1], c=all_colors, s=80, alpha=0.8)
axes[0, 1].set_title("UMAP - All Labels")
axes[0, 1].set_xlabel("UMAP Dimension 1")
axes[0, 1].set_ylabel("UMAP Dimension 2")

axes[0, 2].scatter(trimap_results_scaled[:, 0], trimap_results_scaled[:, 1], c=all_colors, s=80, alpha=0.8)
axes[0, 2].set_title("TriMap - All Labels")
axes[0, 2].set_xlabel("TriMap Dimension 1")
axes[0, 2].set_ylabel("TriMap Dimension 2")

# Second row: "No Finding" vs "Abnormal"
axes[1, 0].scatter(pca_results_scaled[:, 0], pca_results_scaled[:, 1], c=binary_colors, s=80, alpha=0.8)
axes[1, 0].set_title("PCA - No Finding vs Abnormal")
axes[1, 0].set_xlabel("PCA Component 1")
axes[1, 0].set_ylabel("PCA Component 2")

axes[1, 1].scatter(umap_results_scaled[:, 0], umap_results_scaled[:, 1], c=binary_colors, s=80, alpha=0.8)
axes[1, 1].set_title("UMAP - No Finding vs Abnormal")
axes[1, 1].set_xlabel("UMAP Dimension 1")
axes[1, 1].set_ylabel("UMAP Dimension 2")

axes[1, 2].scatter(trimap_results_scaled[:, 0], trimap_results_scaled[:, 1], c=binary_colors, s=80, alpha=0.8)
axes[1, 2].set_title("TriMap - No Finding vs Abnormal")
axes[1, 2].set_xlabel("TriMap Dimension 1")
axes[1, 2].set_ylabel("TriMap Dimension 2")

# Third row: Disease Domains
axes[2, 0].scatter(filtered_pca_results[:, 0], filtered_pca_results[:, 1], c=domain_colors, s=80, alpha=0.8)
axes[2, 0].set_title("PCA - Disease Domains")
axes[2, 0].set_xlabel("PCA Component 1")
axes[2, 0].set_ylabel("PCA Component 2")

axes[2, 1].scatter(filtered_umap_results[:, 0], filtered_umap_results[:, 1], c=domain_colors, s=80, alpha=0.8)
axes[2, 1].set_title("UMAP - Disease Domains")
axes[2, 1].set_xlabel("UMAP Dimension 1")
axes[2, 1].set_ylabel("UMAP Dimension 2")

axes[2, 2].scatter(filtered_trimap_results[:, 0], filtered_trimap_results[:, 1], c=domain_colors, s=80, alpha=0.8)
axes[2, 2].set_title("TriMap - Disease Domains")
axes[2, 2].set_xlabel("TriMap Dimension 1")
axes[2, 2].set_ylabel("TriMap Dimension 2")

# First row: Add legend for all 14 categories
handles_all = [
    plt.Line2D([0], [0], marker='o', color='w', label=category, markersize=10, markerfacecolor=cmap_all(idx / len(original_categories)))
    for idx, category in enumerate(original_categories)
]
axes[0, 2].legend(handles=handles_all, bbox_to_anchor=(1.05, 1), loc='upper left')

# Second row: Add legend for "No Finding" vs. "Abnormal"
handles_binary = [
    plt.Line2D([0], [0], marker='o', color='w', label="No Finding", markersize=10, markerfacecolor="green"),
    plt.Line2D([0], [0], marker='o', color='w', label="Abnormal", markersize=10, markerfacecolor="red")
]
axes[1, 2].legend(handles=handles_binary, bbox_to_anchor=(1.05, 1), loc='upper left')

# Third row: Legend for Disease Domains (Already Present)
handles_domains = [
    plt.Line2D([0], [0], marker='o', color='w', label=domain, markersize=10, markerfacecolor=cmap_domain(idx))
    for idx, domain in enumerate(domain_categories.keys())
]
axes[2, 2].legend(handles=handles_domains, bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout
plt.tight_layout(pad=0)
plt.savefig('finetune_biomedCLIP_embeddingplot.png')


In [ ]:
# class IUXrayDataset(Dataset):
#     """Torch Dataset wrapper for IU‑Xray frontal images."""

#     def __init__(self, df: pd.DataFrame, image_dir: Path, preprocess, labels: list[str]):
#         self.df = df.reset_index(drop=True)
#         self.root = Path(image_dir)
#         self.preprocess = preprocess
#         self.labels = labels

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img_path = self.root / row["filename"]
#         image = self.preprocess(Image.open(img_path).convert("RGB"))
#         # multi‑label → torch float32 multi‑hot
#         y = torch.tensor(row[self.labels].values.astype("float32"))
#         return image, y
    
    
class IUXrayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir, preprocess):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self.image_dir / row['filename']
        img = self.preprocess(Image.open(path).convert('RGB'))
        label = row[classes].values.astype(np.float32)
        return img, label, str(path)

train_loader = DataLoader(IUXrayDataset(train_df_iuxray, image_dir, biomedclip_preprocess),batch_size=24, shuffle=True, num_workers=4)
val_loader = DataLoader(IUXrayDataset(val_df_iuxray, image_dir, biomedclip_preprocess),batch_size=24, shuffle=False, num_workers=4)
test_loader = DataLoader(IUXrayDataset(test_df_iuxray, image_dir, biomedclip_preprocess),batch_size=24, shuffle=False, num_workers=4)


In [ ]:
# def run_zero_shot(test_df: pd.DataFrame, image_dir: Path, preprocess, model, tokenizer):
#     image_paths = [str(image_dir/ p) for p in test_df['filename']]
#     img_emb = extract_image_embeddings(image_paths, preprocess, model, pretrained=False)
#     txt_emb = extract_text_embeddings(classes, tokenizer, model, pretrained=False)
#     logits = img_emb @ txt_emb.T  # (N,14)
#     return logits.numpy()

def run_zero_shot(test_df: pd.DataFrame, image_dir: Path, preprocess, model, tokenizer):
    image_paths = [str(image_dir / p) for p in test_df['filename']]
    img_emb = extract_image_embeddings(image_paths, preprocess, model, pretrained=False)
    txt_emb = extract_text_embeddings(classes, tokenizer, model, pretrained=False)
    logits = img_emb @ txt_emb.T  # (N,14)
    return logits.numpy(), img_emb.numpy(),txt_emb.numpy()

# zs_logits = run_zero_shot(test_df_iuxray, Path(image_dir), biomedclip_preprocess, biomedclip_model, biomedclip_tokenizer)
zs_logits, img_emb, txt_emb  = run_zero_shot(test_df_iuxray, Path(image_dir), biomedclip_preprocess, biomedclip_model, biomedclip_tokenizer)           

In [ ]:
test_df_iuxray = expand_multilabel_fixed(test_df_iuxray)
y_true = test_df_iuxray[classes].values.astype(int)

In [ ]:
ths = np.arange(0.20, 0.91, 0.05)
f1_scores, aucs, lraps, cov_errors = [], [], [], []

for t in ths:
    metric=compute_metrics_v2(zs_logits, y_true, t)
    f1,lrap_score, cov_err= metric['macro_f1'],metric['lrap'], metric['coverage_error']
    print(lrap_score)
    f1_scores.append(f1)
#     aucs.append(auc)
    lraps.append(lrap_score)
    cov_errors.append(cov_err)

# ───────────────  PLOT  ───────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# axes[0, 0].plot(ths, aucs, marker='o')
# axes[0, 0].set_title("Macro-AUROC (higher is better)")
# axes[0, 0].set_xlabel("threshold"); axes[0, 0].set_ylabel("AUROC")

axes[0].plot(ths, f1_scores, marker='o')
axes[0].set_title("Macro-F1 (higher is better)")
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("F1")

axes[1].plot(ths, lraps, marker='o')
axes[1].set_title("LRAP (higher is better)")
axes[1].set_xlabel("threshold"); axes[1].set_ylabel("LRAP")

axes[2].plot(ths, cov_errors, marker='o')
axes[2].set_title("Coverage Error (lower is better)")
axes[2].set_xlabel("threshold"); axes[2].set_ylabel("coverage error")

plt.tight_layout()
plt.show()

#### CHECK MULTI-CLASS ACC

In [ ]:
threshold = 0.5
metrics_v3 = compute_metrics_v4(zs_logits, y_true, threshold, classes)

for disease, stats in metrics_v3.items():
    print(f"{disease:25s}  "
          f"P={stats['precision']:.3f}  "
          f"R={stats['recall']:.3f}  "
          f"F1={stats['f1']:.3f}")


In [ ]:
# def custom_inference(img_path, model,preprocess,text_emb,device):
#     """Return `OrderedDict {label: probability}` for **one** image.

#     • Moves both the image tensor and the cached `text_emb` to the
#       specified device (defaults to the model’s own device).
#     • Applies sigmoid to the similarity scores.
#     • Ensures the final tensor is on CPU before calling `.numpy()`.
#     """

#     # Ensure text embeddings reside on the same device only once
#     txt = torch.as_tensor(text_emb, dtype=torch.float32, device=device)

#     img_emb = (
#         extract_image_embeddings([str(img_path)], preprocess, model, pretrained=False)
#         .to(device)
#     )  # (1, D)

#     logits = (img_emb @ txt.T).cpu().numpy()  # (1,14) on CPU for _row_minmax
#     probs  = expit(logits)[0]           # (14,)
#     sorted_pairs = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)
#     return OrderedDict(sorted_pairs)
# #     return OrderedDict((lbl, float(p)) for lbl, p in zip(classes, probs))

from collections import OrderedDict
import torch, numpy as np

def custom_inference(img_path, model, preprocess, text_emb, device,
                     method="cosine", tau=50.0):
    """
    method = "cosine"   → probs = (cos_sim + 1)/2
             "sigmoid"  → probs = sigmoid(logit / tau)
             "minmax"   → per-row min-max (old behaviour)
    """

    txt = torch.as_tensor(text_emb, dtype=torch.float32, device=device)  # (14,D)

    img_emb = extract_image_embeddings([str(img_path)],
                                       preprocess, model, pretrained=False).to(device)

    # --- compute similarity matrix (1,14) -------------------------------
    logits = img_emb @ txt.T               # (1,14)

    if method == "cosine":
        img_n  = torch.nn.functional.normalize(img_emb, dim=-1)
        txt_n  = torch.nn.functional.normalize(txt, dim=-1)
        logits = img_n @ txt_n.T           # cos ∈ [-1,1]
        probs  = ((logits + 1) / 2).cpu().numpy()[0]

    elif method == "sigmoid":
        logits = logits / tau
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    elif method == "minmax":
        arr    = logits.cpu().numpy()
        mn, mx = arr.min(), arr.max()
        probs  = ((arr - mn) / (mx - mn + 1e-8))[0]

    else:
        raise ValueError("method must be 'cosine', 'sigmoid', or 'minmax'")

    sorted_pairs = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)
    return OrderedDict(sorted_pairs)

idx=111
print('True Label: ',test_df_iuxray['target'][idx])
sample_path = Path(image_dir) / test_df_iuxray['filename'][idx]

all_prob_for_sample=custom_inference(sample_path, biomedclip_model,
                       biomedclip_preprocess, txt_emb, device,
                       method="cosine") 
# all_prob_for_sample = custom_inference(sample_img_path, biomedclip_model, biomedclip_preprocess, txt_emb,device)
print(all_prob_for_sample)

In [ ]:
# df = test_df_iuxray            # or train_df_iuxray if test must stay untouched

# # keep rows that have at least one positive label
# has_label = df[df[classes].sum(1) > 0]

# # sample up to 200 rows, stratified by ‘target’ length if you like
# subset_df = has_label.sample(n=min(200, len(has_label)),
#                              random_state=0)

# # list of (img_path , cls_idx) pairs for evaluation
# validation_subset = [
#     (Path(image_dir) / row.filename,
#      classes.index(row.target[0]))      # first label in list
#     for _, row in subset_df.iterrows()
]
# import numpy as np
# layers = [  vit.trunk.blocks[i].attn.proj  for i in range(12) ]  # ViT-B/16
# scores = []
# # --------------------------------------------------------------------
# # 1.  Make the text-prompt matrix a tensor once, on the correct device
# # --------------------------------------------------------------------
# txt_emb_tensor = torch.as_tensor(
#     txt_emb, dtype=torch.float32, device=device)        # (14, D)
# txt_emb_tensor = torch.nn.functional.normalize(txt_emb_tensor, dim=-1)

# # --------------------------------------------------------------------
# # 2.  Function that returns a scalar similarity for ONE (img, class)
# # --------------------------------------------------------------------
# @torch.no_grad()
# def model_similarity(inp_tensor: torch.Tensor, cls_idx: int) -> float:
#     """
#     inp_tensor : (1, 3, H, W) already on `device`
#     cls_idx    : 0-based index into `classes`
#     Returns    : cosine similarity in [-1,1] as Python float
#     """
#     img_emb = vit(inp_tensor)                                   # (1, D)
#     img_emb = torch.nn.functional.normalize(img_emb, dim=-1)
#     sim = (img_emb @ txt_emb_tensor[cls_idx].unsqueeze(1)).item()
#     return sim


# for layer in layers:
#     cam = GradCAMCustom(model=vit,
#                         target_layers=[layer],
#                         reshape_transform=vit_reshape  # your grid helper
#                         )

#     del_auc_all = []
#     for img_path, cls_idx in validation_subset:          # ~200 images
#         orig, inp   = preprocess_for_cam(img_path)
#         cam_map     = cam(inp, targets=[SimilarityTarget(cls_idx, txt_emb_tensor)])[0]
#         prob_init   = model_similarity(inp, cls_idx)     # cosine or sigmoid

#         # --- deletion curve --------------------------------------------
#         flat_idx = cam_map.flatten().argsort()[::-1]          # hottest → coolest
#         H, W     = cam_map.shape
#         mask_np  = np.ones((H, W), dtype=bool)

#         probs = []
#         for frac in np.linspace(0, 1, 20):
#             k = int(frac * flat_idx.size)
#             mask_np[:] = True
#             mask_np.flat[flat_idx[:k]] = False                # keep previous pixels masked

#             # ---- convert to torch.bool on the same device --------------------
#             mask_t = torch.from_numpy(mask_np).to(masked_img.device)

#             masked_img = inp.clone()
#             # broadcast grey (0.5) into all three channels where mask is False
#             masked_img[0, :, ~mask_t] = 0.5                  # shape-safe indexing

#             probs.append(model_similarity(masked_img, cls_idx))
#         auc = np.trapz(prob_curve, x=np.linspace(0,1,20))
#         del_auc_all.append(auc)
#         if hasattr(cam, "remove_hooks"):
#             cam.remove_hooks()
#         elif hasattr(cam, "clear_hooks"):
#             cam.clear_hooks()

#     scores.append(np.mean(del_auc_all))

# best_layer = layers[int(np.argmin(scores))]
# print("Layer with lowest mean Deletion-AUC:", best_layer)


#### VISUALIZE A SAMPLE IMAGE PER CATEGORY WITH GRADCAM AND LAYER -3

In [ ]:
txt_emb_tensor = torch.as_tensor(txt_emb, device=device)

In [ ]:
txt_emb_tensor = torch.as_tensor(txt_emb, device=device)
for disease in all_prob_for_sample:
    cls_idx = classes.index(disease)   
    plot_gradcam(sample_path,disease,cls_idx,txt_emb_tensor,all_prob_for_sample[disease])

#### CHECK EMBEDDING DISTANCES

In [ ]:
probe_acc, knn_acc, sil, db, fisher, intra_d, inter_d, ic_ia = compute_separability(zs_emb, y_true)
print(probe_acc, knn_acc, sil, db, fisher, intra_d, inter_d, ic_ia)

In [ ]:
from typing import Dict, List
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

# ------------------------- 1. Mapping ----------------------------------
domain_categories = {
    "Cardiovascular": ["Enlarged Cardiomediastinum", "Cardiomegaly"],
    "Pulmonary": ["Pneumonia", "Consolidation", "Atelectasis", "Pneumothorax",
                  "Pleural Other", "Pleural Effusion", "Edema",
                  "Lung Opacity", "Lung Lesion"],
    "Skeletal": ["Fracture"],
    "Device": ["Support Devices"],
}

# Make index lists once
cls2idx   = {c: i for i, c in enumerate(classes)}     # classes = 14-label list
domain2col = {
    dom: np.array([cls2idx[d] for d in diseases], dtype=int)
    for dom, diseases in domain_categories.items()
}

# ------------------------- 2. Domain-level metric ----------------------
def compute_domain_metrics(logits: np.ndarray,
                           y_true: np.ndarray,
                           threshold: float,
                           prob_mode: str = "cosine",
                           tau: float = 80.0) -> Dict[str, Dict[str, float]]:
    """
    Collapse per-disease logits & labels into 4 domains and
    return {domain: {'precision', 'recall', 'f1'}}.
    """

    # a) probabilities for the 14 diseases
    y_prob14 = logits_to_prob(logits, mode=prob_mode, tau=tau)
    y_pred14 = (y_prob14 > threshold).astype(int)

    # b) aggregate to 4-column arrays (N, 4)
    N = logits.shape[0]
    y_prob_dom = np.zeros((N, 4), dtype=float)
    y_true_dom = np.zeros((N, 4), dtype=int)
    for j, (dom, cols) in enumerate(domain2col.items()):
        y_prob_dom[:, j] = y_prob14[:, cols].max(1)       # OR via max-prob
        y_true_dom[:, j] = (y_true[:,  cols].sum(1) > 0).astype(int)

    y_pred_dom = (y_prob_dom > threshold).astype(int)

    # c) per-domain P/R/F1
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_dom, y_pred_dom, average=None, zero_division=0)

    return {
        dom: {"precision": float(p), "recall": float(r), "f1": float(f)}
        for dom, p, r, f in zip(domain_categories.keys(), prec, rec, f1)
    }


In [ ]:
domain_metrics = compute_domain_metrics(
    zs_logits, y_true, threshold=0.5, prob_mode="cosine")

for dom, m in domain_metrics.items():
    print(f"{dom:15s}  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}")

##### FINETUNING - ENTIRE NETWORK vs. LINEAR PROBE

In [ ]:
cls2idx = {c: i for i,c in enumerate(classes)}
# --------------------------------------------------------------------------
# 1. Dataset
# --------------------------------------------------------------------------
class IUXrayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir, preprocess):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = self.image_dir / row['filename']
        img  = self.preprocess(Image.open(path).convert('RGB'))
        label= row[classes].values.astype(np.float32)
        return img, torch.from_numpy(label), str(path)

for df in (train_df_iuxray, val_df_iuxray, test_df_iuxray):
        expand_multilabel_fixed(df)

bs = 24
train_loader = DataLoader(IUXrayDataset(train_df_iuxray, image_dir, biomedclip_preprocess),
                          batch_size=bs, shuffle=True,  num_workers=4,
                          persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(IUXrayDataset(val_df_iuxray,   image_dir, biomedclip_preprocess),
                          batch_size=bs, shuffle=False, num_workers=4,
                          persistent_workers=True, pin_memory=True)
test_loader  = DataLoader(IUXrayDataset(test_df_iuxray,  image_dir, biomedclip_preprocess),
                          batch_size=bs, shuffle=False, num_workers=4,
                          persistent_workers=True, pin_memory=True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:

# 2. CLIP vision tower + 14‑logit head
# --------------------------------------------------------------------------
class BioMedCLIPClassifier(nn.Module):
    def __init__(self, clip_model, n_classes=14):
        super().__init__()
        self.visual = clip_model.visual   # ViT
        self.head   = nn.Linear(512, n_classes)
        nn.init.xavier_uniform_(self.head.weight)
    def forward(self, x):
        feat = self.visual(x)             # (B,512)
        return self.head(feat)            # (B,14)

# --------------------------------------------------------------------------
# 3. Epoch helpers
# --------------------------------------------------------------------------
@torch.no_grad()
def _inference(model, loader, device):
    model.eval(); losses=[]; y_true=[]; y_lgt=[]
    for imgs, labels, _ in tqdm(loader, leave=False):
        imgs=imgs.to(device); labels=labels.to(device)
        logits = model(imgs)
        loss   = F.binary_cross_entropy_with_logits(logits, labels)
        losses.append(loss.item())
        y_true.append(labels.cpu().numpy())
        y_lgt.append(logits.cpu().numpy())
    return np.mean(losses), np.vstack(y_true), np.vstack(y_lgt)

def _train_epoch(model, loader, optim, scaler, device):
    model.train(); losses=[]
    for imgs, labels, _ in tqdm(loader, leave=False):
        imgs=imgs.to(device); labels=labels.to(device)
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            logits = model(imgs)
            loss   = F.binary_cross_entropy_with_logits(logits, labels)
        scaler.scale(loss).backward() if scaler else loss.backward()
        scaler.step(optim); scaler.update() if scaler else optim.step()
        losses.append(loss.item())
    return np.mean(losses)

# --------------------------------------------------------------------------
# 4. Metrics helpers (simple)
# --------------------------------------------------------------------------
from scipy.special import expit

def logits_to_prob(logits, mode="sigmoid", tau=80.):
    if mode=="sigmoid": return expit(logits/tau)
    mn=logits.min(1,keepdims=True);mx=logits.max(1,keepdims=True)
    return (logits-mn)/(mx-mn+1e-8)

def compute_domain_metrics(logits, y_true, threshold=0.5, prob_mode="sigmoid", tau=80.):
    domains={
        "Cardiovascular":[0,1],
        "Pulmonary"    :[2,3,4,5,6,7,8,9,10],
        "Skeletal"     :[11],
        "Device"       :[12]
    }
    y_prob=logits_to_prob(logits,mode=prob_mode,tau=tau)
    y_pred=(y_prob>threshold).astype(int)
    out={}
    for dom,idxs in domains.items():
        true_any = (y_true[:,idxs].sum(1)>0).astype(int)
        pred_any = (y_pred[:,idxs].sum(1)>0).astype(int)
        p,r,f,_=precision_recall_fscore_support(true_any,pred_any,average='binary',zero_division=0)
        out[dom]={"precision":float(p),"recall":float(r),"f1":float(f)}
    return out

def compute_metrics_v4(logits,y_true,threshold,class_names,prob_mode="sigmoid",tau=80.):
    y_prob=logits_to_prob(logits,mode=prob_mode,tau=tau)
    y_pred=(y_prob>threshold).astype(int)
    p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average=None,zero_division=0)
    return{c:{"precision":float(pp),"recall":float(rr),"f1":float(ff)}for c,pp,rr,ff in zip(class_names,p,r,f)}



# ------------------------------------------------------------------
# Training + evaluation driver  (returns history)
# ------------------------------------------------------------------
def train_eval_biomedclip(train_loader, val_loader, test_loader,
                          clip_model, *,
                          mode="linearprobe",        # or "finetune"
                          n_epochs=8, lr=1e-4,
                          warmup_epochs=1, patience=3,
                          ckpt_path="best_clip.pth", device="cuda"):
    """Train BioMedCLIP classifier – returns (test_logits, history_dict)."""

    device = torch.device(device)
    model  = BioMedCLIPClassifier(clip_model).to(device)

    # freeze vision tower initially
    for p in model.visual.parameters():
        p.requires_grad_(False)

    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    optim  = torch.optim.AdamW(filter(lambda p: p.requires_grad_, model.parameters()),
                               lr=lr, weight_decay=1e-2)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=n_epochs)

    best_val, patience_ctr = float("inf"), 0
    history = {"train_loss": [], "val_loss": [], "val_macroF1": []}

    for epoch in range(1, n_epochs + 1):
        # un‑freeze after warm‑up for fine‑tune
        if mode == "finetune": #and epoch == warmup_epochs + 1:
            for p in model.visual.parameters():
                p.requires_grad_(True)
            optim = torch.optim.AdamW(model.parameters(),
                                      lr=lr * 0.1, weight_decay=1e-2)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optim, T_max=n_epochs - warmup_epochs)
            print("\n→ Vision tower unfrozen (fine‑tune phase)")

        print(f"\nEpoch {epoch}/{n_epochs}  |  LR {sched.get_last_lr()[0]:.2e}")
        tr_loss = _train_epoch(model, train_loader, optim, scaler, device)
        val_loss, y_val, lgt_val = _inference(model, val_loader, device)
        sched.step()

        # compute val macro‑F1 for logging purposes
        val_macro = np.mean([d["f1"] for d in
                     compute_metrics_v4(lgt_val, y_val, 0.5, classes).values()])

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_macroF1"].append(val_macro)
        print(f"  train BCE {tr_loss:.4f}   val BCE {val_loss:.4f}   "
              f"val macro‑F1 {val_macro:.3f}")
        if val_loss < best_val - 1e-4:
            best_val = val_loss; patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
            print("  ✔ checkpoint saved")
        else:
            patience_ctr += 1
            print(f"  no improvement ({patience_ctr}/{patience})")
            if patience_ctr >= patience:
                print("Early stopping triggered."); break

    # -------- load best & test --------
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    _, y_test, lgt_test = _inference(model, test_loader, device)

    # -------- final metrics ----------
    print(f"\n=== Test metrics (best val BCE {best_val:.4f}) ===")
    per_lbl = compute_metrics_v4(lgt_test, y_test, 0.5, classes)
    for lbl,d in per_lbl.items():
        print(f"{lbl:25s} P={d['precision']:.3f} R={d['recall']:.3f} F1={d['f1']:.3f}")

    per_dom = compute_domain_metrics(lgt_test, y_test, 0.5)
    print("\n-- per domain --")
    for dom,d in per_dom.items():
        print(f"{dom:15s} P={d['precision']:.3f} R={d['recall']:.3f} F1={d['f1']:.3f}")

    return lgt_test, history


In [ ]:
warmup_steps=1

In [ ]:
n_epochs=10
num_training_steps = (len(train_loader) // bs) * n_epochs
warmup_steps = int(0.1 * num_training_steps) 

#### FULL FINE TUNING

In [ ]:

# ----- full fine‑tune ------------------------------------------------------
logits_ft,hist_ft = train_eval_biomedclip(
    train_loader, val_loader, test_loader,
    biomedclip_model,
    mode="finetune",             # unfreeze after 1 epoch
    n_epochs=n_epochs
    , lr=0.0001,
    warmup_epochs=warmup_steps, patience=5,
    ckpt_path="best_ft_v2.pth", device="cuda")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(hist_ft["train_loss"], label="train loss")
plt.plot(hist_ft["val_loss"],   label="val loss")
# plt.twinx()
# plt.plot(hist_ft["val_macroF1"], color="green", label="val macro‑F1")
plt.legend(); plt.xlabel("epoch"); plt.show()

In [ ]:
# ---------------------------------------------------------------
# 1.  Load best model (vision‑tower + head)
# ---------------------------------------------------------------
device = "cuda"
model_ft = BioMedCLIPClassifier(biomedclip_model).to(device)
model_ft.load_state_dict(torch.load("best_ft.pth", map_location=device))
model_ft.eval()                     # inference mode

# ---------------------------------------------------------------
# 2.  Helper to grab embeddings + labels
# ---------------------------------------------------------------
@torch.no_grad()
def extract_vit_embeddings(model, loader, device="cuda"):
    feats, labels = [], []
    for imgs, lbl, _ in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        # we want the **vision‑tower output before the head**
        f = model.visual(imgs)              # (B, 512)
        feats.append(f.cpu())
        labels.append(lbl)                  # already on CPU
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

emb_ft, y_true = extract_vit_embeddings(model_ft, test_loader, device)

# ---------------------------------------------------------------
# 3.  Compute separability scores
# ---------------------------------------------------------------
probe_acc, knn_acc, sil, db, fisher, intra_d, inter_d, ic_ia = (
    compute_separability(emb_ft, y_true)
)

print(f"LogReg probe acc  : {probe_acc:.3f}")
print(f"k‑NN (k=5) acc    : {knn_acc:.3f}")
print(f"Silhouette        : {sil:.3f}   (higher = better)")
print(f"Davies–Bouldin    : {db:.3f}   (lower = better)")
print(f"Fisher ratio      : {fisher:.3f}")
print(f"Intra‑class dist  : {intra_d:.3f}")
print(f"Inter‑class dist  : {inter_d:.3f}")
print(f"Inter / Intra     : {ic_ia:.3f}")


In [ ]:
_, y_test, lgt_test = _inference(model_ft, test_loader, "cuda")

ft_metrics = compute_metrics_v2(lgt_test, y_test, 0.5)
print(ft_metrics)

In [ ]:
from copy import deepcopy
import numpy as np
from sklearn.metrics import f1_score

def train_val_only(train_loader, val_loader,
                   clip_model,
                   mode="linearprobe",
                   n_epochs=3,
                   lr=1e-4,
                   warmup_epochs=1,
                   patience=2,
                   ckpt_path="tmp.pth",
                   device="cuda"):
    """
    Exactly the same training loop as `train_eval_biomedclip`,
    but only returns validation labels & logits at the end.
    """
    # make a fresh copy so we don’t clobber your real checkpoint
    model = BioMedCLIPClassifier(clip_model).to(device)
    # freeze visual backbone initially
    for p in model.visual.parameters(): p.requires_grad_(False)

    optim = torch.optim.AdamW(
        filter(lambda p: p.requires_grad_, model.parameters()),
        lr=lr, weight_decay=1e-2
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=n_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device!="cpu"))

    best_val_loss = float("inf")
    epochs_no_imp = 0

    for epoch in range(1, n_epochs+1):
        # unfreeze if fine-tuning
        if mode=="finetune" and epoch==warmup_epochs+1:
            for p in model.visual.parameters(): p.requires_grad_(True)
            optim = torch.optim.AdamW(model.parameters(), lr=lr*0.1, weight_decay=1e-2)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                optim, T_max=n_epochs-warmup_epochs
            )

        # ---- train epoch ----
        model.train()
        total_train = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optim.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device!="cpu")):
                logits = model(imgs)
                loss   = F.binary_cross_entropy_with_logits(logits, labels)
            if scaler: scaler.scale(loss).backward()
            else:       loss.backward()
            if scaler:
                scaler.step(optim); scaler.update()
            else:
                optim.step()
            total_train += loss.item()
        # ---- val epoch ----
        model.eval()
        total_val = 0.0
        all_y, all_l = [], []
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss   = F.binary_cross_entropy_with_logits(logits, labels)
                total_val += loss.item()
                all_y.append(labels.cpu().numpy())
                all_l.append(logits.cpu().numpy())
        val_loss = total_val/len(val_loader)
        y_val    = np.vstack(all_y)
        lgt_val  = np.vstack(all_l)

        # early-stop on val_loss
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            epochs_no_imp = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_imp += 1
            if epochs_no_imp >= patience:
                break
        sched.step()

    # reload best
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return y_val, lgt_val


def tune_lr_and_threshold(
    train_loader, val_loader,
    clip_model,
    lr_candidates,    # e.g. [1e-5,3e-5,1e-4]
    thr_candidates,   # e.g. np.linspace(0.1,0.9,17)
    **train_val_kwargs
):
    best_score = -1.0
    best_cfg   = {}
    for lr in lr_candidates:
        # **IMPORTANT**: use num_workers=0 inside tuning
        train_loader.num_workers = val_loader.num_workers = 0

        y_val, logits_val = train_val_only(
            train_loader, val_loader,
            clip_model,
            lr=lr,
            **train_val_kwargs
        )

        # compute probs once via your favorite mode
        probs_val = logits_to_prob(logits_val, mode="sigmoid", tau=80.0)

        for thr in thr_candidates:
            y_pred = (probs_val > thr).astype(int)
            # flatten across all classes & samples
            score = f1_score(y_val.ravel(), y_pred.ravel(),
                             average="macro", zero_division=0)
            if score > best_score:
                best_score = score
                best_cfg   = {"lr": lr, "threshold": thr}

    return best_cfg, best_score


# ---- call it ----
lr_grid  = [1e-5, 3e-5, 1e-4]
thr_grid = np.linspace(0.1, 0.9, 17)
best_cfg, best_f1 = tune_lr_and_threshold(
    train_loader, val_loader,
    clip_model=biomedclip_model,
    lr_candidates=lr_grid,
    thr_candidates=thr_grid,
    n_epochs=3,
    mode="finetune",
    warmup_epochs=1,
    patience=2
)


In [ ]:
print("→ Best on val:", best_cfg, "macro-F1=", best_f1)

In [ ]:
warmup_steps=1

#### LINEAR PROBING

In [ ]:
biomedclip_model, biomedclip_preprocess,biomedclip_tokenizer=initializeModel()

In [ ]:
 # ----- linear‑probe --------------------------------------------------------
logits_lp,hist_lp = train_eval_biomedclip(
    train_loader, val_loader, test_loader,
    biomedclip_model,
    mode="linearprobe",          # freeze vision tower
    n_epochs=n_epochs
    , lr=0.0001,
    warmup_epochs=warmup_steps, patience=5,
    ckpt_path="best_lp_v2.pth", device="cuda")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(hist_lp["train_loss"], label="train loss")
plt.plot(hist_lp["val_loss"],   label="val loss")
# plt.twinx()
# plt.plot(hist_ft["val_macroF1"], color="green", label="val macro‑F1")
plt.legend(); plt.xlabel("epoch"); plt.show()

In [ ]:
# ---------------------------------------------------------------
# 1.  Load best model (vision‑tower + head)
# ---------------------------------------------------------------
device = "cuda"
model_lp = BioMedCLIPClassifier(biomedclip_model).to(device)
model_lp.load_state_dict(torch.load("best_lp_v2.pth", map_location=device))
model_lp.eval()                     # inference mode

# ---------------------------------------------------------------
# 2.  Helper to grab embeddings + labels
# ---------------------------------------------------------------
@torch.no_grad()
def extract_vit_embeddings(model, loader, device="cuda"):
    feats, labels = [], []
    for imgs, lbl, _ in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        # we want the **vision‑tower output before the head**
        f = model.visual(imgs)              # (B, 512)
        feats.append(f.cpu())
        labels.append(lbl)                  # already on CPU
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

emb_lp, y_true = extract_vit_embeddings(model_lp, test_loader, device)

# ---------------------------------------------------------------
# 3.  Compute separability scores
# ---------------------------------------------------------------
probe_acc, knn_acc, sil, db, fisher, intra_d, inter_d, ic_ia = (
    compute_separability(emb_lp, y_true)
)

print(f"LogReg probe acc  : {probe_acc:.3f}")
print(f"k‑NN (k=5) acc    : {knn_acc:.3f}")
print(f"Silhouette        : {sil:.3f}   (higher = better)")
print(f"Davies–Bouldin    : {db:.3f}   (lower = better)")
print(f"Fisher ratio      : {fisher:.3f}")
print(f"Intra‑class dist  : {intra_d:.3f}")
print(f"Inter‑class dist  : {inter_d:.3f}")
print(f"Inter / Intra     : {ic_ia:.3f}")


In [ ]:
_, y_test, lgp_test = _inference(model_lp, test_loader, "cuda")


lp_metrics = compute_metrics_v2(lgp_test, y_test, 0.5)
print(lp_metrics)

#### GRAD CAM VIZ FOR THREE CONFIGS

In [ ]:
zs_model, zs_preprocess,zs_tokenizer=initializeModel()
zs_model.eval()

model_ft = BioMedCLIPClassifier(zs_model).to(device)
model_ft.load_state_dict(torch.load("best_ft_v2.pth", map_location=device))
model_ft.eval()                     # inference mode

zs_model, zs_preprocess,zs_tokenizer=initializeModel()
zs_model.eval()


model_lp = BioMedCLIPClassifier(zs_model).to(device)
model_lp.load_state_dict(torch.load("best_lp_v2.pth", map_location=device))
model_lp.eval()                     # inference mode



def returnVitLayer(model):
    return model.visual.to(device).eval()
zs_vit = returnVitLayer(biomedclip_model)
ft_vit = returnVitLayer(model_ft)
lp_vit = returnVitLayer(model_lp)

# ─────────────────────  2. TARGET LAYER FOR GRAD-CAM  ──────────────────
# Use the projection of the last attention block

def targetlayer(vit,N):
    return vit.trunk.blocks[-N].attn.proj
zs_target_layer = targetlayer(zs_vit,3)
ft_target_layer = targetlayer(ft_vit,3)
lp_target_layer = targetlayer(lp_vit,3)

# ─────────────────────  3. CUSTOM SCALAR TARGET  ───────────────────────
class EmbeddingNormTarget:
    """Return ‖embedding‖₂ so Grad-CAM has a scalar loss."""
    def __call__(self, model_output):
        return model_output.norm()


def vit_reshape(x, h=14, w=14):
    """
    Convert (B, N, C) ViT tokens to (B, C, H, W).
    CLS token is dropped, remainder reshaped to grid.
    """
    B, N, C = x.shape            # N = 1 + h*w
    x = x[:, 1:, :].permute(0, 2, 1)              # (B, C, h*w)
    return x.reshape(B, C, h, w)

class SimilarityTarget:
    """Scalar = similarity(image emb, prompt[cls_idx])."""
    def __init__(self, cls_idx: int, txt_mat: torch.Tensor):
        self.cls = cls_idx                  # int 0-13
        self.txt = txt_mat                  # (14, D) tensor on same device

    def __call__(self, img_emb: torch.Tensor):
        # img_emb is (D,) or (1, D) depending on torch-cam version
        if img_emb.dim() == 1:              # (D,)
            return torch.dot(img_emb, self.txt[self.cls])
        else:                               # (1, D)
            return (img_emb @ self.txt.T)[0, self.cls]

## ─── 5. Grad-CAM wrapper that handles tensor output ─────────────────────
class GradCAMCustom(GradCAM):
    def forward(self, x, targets, eigen_smooth=False):
        out  = self.model(x)                      # img_emb (B, D)
        loss = sum(t(out) for t in targets)
        loss.backward(retain_graph=True)
        return super().forward(x, targets, eigen_smooth)

zs_cam = GradCAMCustom(
        model=zs_vit,
        target_layers=[zs_target_layer],
        reshape_transform=vit_reshape
)

ft_cam = GradCAMCustom(
        model=ft_vit,
        target_layers=[ft_target_layer],
        reshape_transform=vit_reshape
)

lp_cam = GradCAMCustom(
        model=lp_vit,
        target_layers=[lp_target_layer],
        reshape_transform=vit_reshape
)

In [ ]:

# ─── 6. Pre-processing / plotting helpers ───────────────────────────────
def preprocess_for_cam(path):
    img = Image.open(path).convert("RGB")
    original = np.asarray(img.resize((224, 224))).astype(np.float32) / 255.0
    tensor   = biomedclip_preprocess(img).unsqueeze(0).to(device)
    return original, tensor

def generate_gradcam(cam,path,cls_idx,txt_emb_tensor):
    orig, inp = preprocess_for_cam(path)
    target_fn = SimilarityTarget(cls_idx, txt_emb_tensor)
    cam_map   = cam(inp, targets=[target_fn])[0]
#     cam_map   = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    if hasattr(cam, "remove_hooks"):
        cam.remove_hooks()
    elif hasattr(cam, "clear_hooks"):
        cam.clear_hooks()
    return orig, show_cam_on_image(orig, cam_map, use_rgb=True)

def plot_gradcam(cam,path,disease,cls_idx,txt_emb_tensor,disease_prob):
    orig, cam_img = generate_gradcam(cam,path,cls_idx,txt_emb_tensor)
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(orig);    ax[0].axis("off"); ax[0].set_title("Original")
    ax[1].imshow(cam_img); ax[1].axis("off"); ax[1].set_title(f"{disease}, Probability: {disease_prob:.3g}",fontsize=18)
    plt.tight_layout(); plt.show()
    if hasattr(cam, "remove_hooks"):
        cam.remove_hooks()
    elif hasattr(cam, "clear_hooks"):
        cam.clear_hooks()


from collections import OrderedDict
import torch, numpy as np

def extract_image_embeddings(image_paths, preprocess, model, device):
    """
    Given a list of image file paths, run them through either
    1) model.encode_image(...)  (for raw CLIP models), or
    2) model.visual(...)        (for BioMedCLIPClassifier),
    to get back a (N, D) tensor of image embeddings on `device`.
    """
    embeds = []
    model = model.to(device)
    model.eval()
    for p in image_paths:
        img = Image.open(p).convert("RGB")
        inp = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            if hasattr(model, "encode_image"):
                e = model.encode_image(inp)        # raw CLIP
            else:
                e = model.visual(inp)              # your classifier
        embeds.append(e.cpu())
    return torch.cat(embeds, dim=0)


def custom_inference(img_path, model, preprocess, text_emb, device,
                     method="cosine", tau=50.0):
    """
    img_path:  Path or str
    model:     either raw CLIP or your BioMedCLIPClassifier
    preprocess:  your biomedclip_preprocess (same as during training)
    text_emb:    only used if model has encode_image; you can pass None
                 when you just want the classifier’s logits.
    """
    # If text_emb is provided, we're in zero-shot mode.  Otherwise
    # we'll just forward through classification head.
    if text_emb is not None and hasattr(model, "encode_image"):
        # Zero-shot style:
        txt = torch.as_tensor(text_emb, dtype=torch.float32, device=device)  # (14, D)
        img_emb = extract_image_embeddings([str(img_path)], preprocess, model, device)  # (1, D) on CPU
        img_emb = img_emb.to(device)
        logits = img_emb @ txt.T                 # (1,14)
        if method == "cosine":
            img_n = F.normalize(img_emb, dim=-1)
            txt_n = F.normalize(txt,   dim=-1)
            logits = img_n @ txt_n.T            # cos ∈ [-1,1]
            probs  = ((logits + 1) / 2).cpu().numpy()[0]
        elif method == "sigmoid":
            probs = torch.sigmoid(logits / tau).cpu().numpy()[0]
        elif method == "minmax":
            arr = logits.cpu().numpy()
            mn, mx = arr.min(), arr.max()
            probs = ((arr - mn) / (mx - mn + 1e-8))[0]
        else:
            raise ValueError
    else:
        # Classification‐head style:
        # text_emb is ignored here; we just forward through .head
        img = Image.open(img_path).convert("RGB")
        inp = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(inp)               # (1,14)
        if method == "sigmoid":
            probs = torch.sigmoid(logits / tau).cpu().numpy()[0]
        elif method == "minmax":
            arr = logits.cpu().numpy()
            mn, mx = arr.min(), arr.max()
            probs = ((arr - mn) / (mx - mn + 1e-8))[0]
        else:
            raise ValueError("For classifier, only 'sigmoid' or 'minmax' supported")
    from collections import OrderedDict
    sorted_pairs = sorted(zip(classes, probs), key=lambda x: x[1], reverse=True)
    return OrderedDict(sorted_pairs)





# #111, 11
# idx=111
# print('True Label: ',test_df_iuxray['target'][idx])
# sample_path = Path(image_dir) / test_df_iuxray['filename'][idx]




# ft_probs = custom_inference(
#     sample_path,
#     ft_model,
#     zs_preprocess,
#     text_emb=None,        # ignored when you just want classifier head
#     device=device,
#     method="sigmoid",
#     tau=50.0
# )

# lp_probs = custom_inference(
#     sample_path,
#     lp_model,
#     zs_preprocess,
#     text_emb=None,
#     device=device,
#     method="sigmoid",
#     tau=50.0
# )

# zs_probs = custom_inference(
#     sample_path,
#     zs_model,
#     zs_preprocess,
#     txt_emb,
#     device=device,
#     method="cosine"
# )



In [ ]:
from pytorch_grad_cam import GradCAM
# … your EmbeddingNormTarget, SimilarityTarget, vit_reshape, generate_gradcam, etc. …

def make_cam(model_visual):
    # pick the same relative layer index for all three
    target_layer = model_visual.trunk.blocks[-3].attn.proj  
    return GradCAMCustom(
      model=model_visual.eval().to(device),
      target_layers=[target_layer],
      reshape_transform=vit_reshape
    )

# build your three CAM wrappers
cam_zs  = GradCAMCustom(
        model=zs_vit,
        target_layers=[zs_target_layer],
        reshape_transform=vit_reshape
)
cam_ft  = GradCAMCustom(
        model=ft_vit,
        target_layers=[ft_target_layer],
        reshape_transform=vit_reshape
)

cam_lp  = GradCAMCustom(
        model=lp_vit,
        target_layers=[lp_target_layer],
        reshape_transform=vit_reshape
)

settings = [
    ("Zero-Shot",   cam_zs, zs_model, zs_preprocess, zs_probs),
    ("Fine-Tuned",  cam_ft, ft_model, zs_preprocess, ft_probs),
    ("LinearProbe", cam_lp, lp_model, zs_preprocess, lp_probs),
]
K=14
fig, axes = plt.subplots(K, 3, figsize=(12, 4*K))
for col, (name, cam, model, preprocess, probs) in enumerate(settings):
    topdiseases = list(probs.keys())[:K]
    for row, disease in enumerate(topdiseases):
        cls_idx    = classes.index(disease)
        disease_p  = probs[disease]
        orig, cam_img = generate_gradcam(
            cam, sample_path, cls_idx, txt_emb_tensor
        )
        ax = axes[row, col]
        ax.imshow(cam_img)
        ax.axis("off")
        ax.set_title(f"{name}\n{disease} (P={disease_p:.2f})")
plt.tight_layout()
plt.show()


In [ ]:
# 1) Sample 15 random rows and write out their filenames
import pandas as pd

# sample 15 rows (set random_state for reproducibility)
subset_df = test_df_iuxray.sample(n=15, random_state=42).reset_index(drop=True)

# (optionally) save the small dataframe for later
subset_df.to_csv("subset_test_df.csv", index=False)

# # write just the file names (one per line) to a txt file
# subset_df["filename"].to_csv("subset_filenames.txt", index=False, header=False)

# # Inspect
# subset_df


In [ ]:
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model_ft.zero_grad()
model_lp.zero_grad()

def returnVitLayer(model):
    return model.visual.to(device).eval()
zs_vit = returnVitLayer(biomedclip_model)
ft_vit = returnVitLayer(model_ft)
lp_vit = returnVitLayer(model_lp)

# ─────────────────────  2. TARGET LAYER FOR GRAD-CAM  ──────────────────
# Use the projection of the last attention block

def targetlayer(vit,N):
    return vit.trunk.blocks[N].attn.proj
zs_target_layer = targetlayer(zs_vit,-1)
ft_target_layer = targetlayer(ft_vit,-1)
lp_target_layer = targetlayer(lp_vit,-1)

# ─────────────────────  3. CUSTOM SCALAR TARGET  ───────────────────────
class EmbeddingNormTarget:
    """Return ‖embedding‖₂ so Grad-CAM has a scalar loss."""
    def __call__(self, model_output):
        return model_output.norm()


def vit_reshape(x, h=14, w=14):
    """
    Convert (B, N, C) ViT tokens to (B, C, H, W).
    CLS token is dropped, remainder reshaped to grid.
    """
    B, N, C = x.shape            # N = 1 + h*w
    x = x[:, 1:, :].permute(0, 2, 1)              # (B, C, h*w)
    return x.reshape(B, C, h, w)

class SimilarityTarget:
    """Scalar = similarity(image emb, prompt[cls_idx])."""
    def __init__(self, cls_idx: int, txt_mat: torch.Tensor):
        self.cls = cls_idx                  # int 0-13
        self.txt = txt_mat                  # (14, D) tensor on same device

    def __call__(self, img_emb: torch.Tensor):
        # img_emb is (D,) or (1, D) depending on torch-cam version
        if img_emb.dim() == 1:              # (D,)
            return torch.dot(img_emb, self.txt[self.cls])
        else:                               # (1, D)
            return (img_emb @ self.txt.T)[0, self.cls]

## ─── 5. Grad-CAM wrapper that handles tensor output ─────────────────────
class GradCAMCustom(GradCAM):
    def forward(self, x, targets, eigen_smooth=False):
        out  = self.model(x)                      # img_emb (B, D)
        loss = sum(t(out) for t in targets)
        loss.backward(retain_graph=True)
        return super().forward(x, targets, eigen_smooth)

zs_cam = GradCAMCustom(
        model=zs_vit,
        target_layers=[zs_target_layer],
        reshape_transform=vit_reshape
)

ft_cam = GradCAMCustom(
  model=model_ft,            # <-- full BioMedCLIPClassifier
  target_layers=[ft_target_layer],
  reshape_transform=vit_reshape
)
lp_cam = GradCAMCustom(
  model=model_lp,            # <-- full BioMedCLIPClassifier
  target_layers=[lp_target_layer],
  reshape_transform=vit_reshape
)

def generate_gradcam_zs(zs_cam,path, cls_idx,txt_emb_tensor):
    orig, inp = preprocess_for_cam(path)
    inp = inp.to(device).requires_grad_(True)
    target_fn = SimilarityTarget(cls_idx, txt_emb_tensor)
    grayscale = zs_cam(inp, targets=[target_fn])[0]
    return orig, show_cam_on_image(orig, grayscale, use_rgb=True)


def generate_gradcam_clf(cam, path, cls_idx):
    orig, inp = preprocess_for_cam(path)
    inp = inp.to(device).requires_grad_(True)
    target_fn = ClassifierOutputTarget(cls_idx)     # now applied to model_ft(inp)
    grayscale = cam(inp, targets=[target_fn])[0]    # <-- grads will exist!
    return orig, show_cam_on_image(orig, grayscale, use_rgb=True)


In [ ]:
import torch
diff_trunk = torch.norm(
    model_ft.visual.trunk.blocks[-3].attn.proj.weight
  - model_lp.visual.trunk.blocks[-3].attn.proj.weight
)
print("Trunk weight ‖Δ‖:", diff_trunk.item())

In [ ]:
filenames = [
    "773_IM-2318-1001.dcm.png",
    "992_IM-2477-0001-0002.dcm.png",
    "1196_IM-0131-1001.dcm.png",
]

sample_path = Path(image_dir) / filenames[2]

ft_probs = custom_inference(
    sample_path,
    model_ft,
    zs_preprocess,
    text_emb=None,        # ignored when you just want classifier head
    device=device,
    method="sigmoid",
    tau=50.0
)

lp_probs = custom_inference(
    sample_path,
    model_lp,
    zs_preprocess,
    text_emb=None,
    device=device,
    method="sigmoid",
    tau=50.0
)

zs_probs = custom_inference(
    sample_path,
    zs_model,
    zs_preprocess,
    txt_emb,
    device=device,
    method="cosine"
)

settings = [
    ("Zero-Shot",   zs_cam, zs_model, zs_preprocess, zs_probs),
    ("Fine-Tuned",  ft_cam, model_ft, zs_preprocess, ft_probs),
    ("LinearProbe", lp_cam, model_lp, zs_preprocess, lp_probs),
]

K = len(classes)   # 14
fig, axes = plt.subplots(K, 3, figsize=(12, 4*K), squeeze=False)

for col, (name, cam, model, preprocess, probs) in enumerate(settings):
    # settings is your list of 3 tuples:
    #   ("Zero-Shot", cam_zs, zs_model, zs_preprocess, zs_probs)
    #   ("Fine-Tuned", cam_ft, ft_model, ft_preprocess, ft_probs)
    #   ("LinearProbe", cam_lp, lp_model, lp_preprocess, lp_probs)

    # we want all 14 diseases, in order
    topdiseases = classes  

    for row, disease in enumerate(topdiseases):
        cls_idx   = classes.index(disease)
        disease_p = probs[disease]

        # generate the Grad-CAM overlay
        if name=='Zero-Shot':
            orig, cam_img = generate_gradcam_zs(
                cam,
                sample_path,      # the one image you picked earlier
                cls_idx,
                txt_emb_tensor
            )
        else:
            orig, cam_img = generate_gradcam_clf(cam, sample_path, cls_idx)
            

        
        ax = axes[row, col]
        ax.imshow(cam_img)
        ax.axis("off")
        ax.set_title(f"{name}\n{disease} (P={disease_p:.2f})")

plt.tight_layout()
plt.show()


In [ ]:


#cosine for zs, sigmoid for ft lp
# all_prob_for_sample=custom_inference(sample_path, ft_model,
                       # zs_preprocess, txt_emb, device,
                       # method="sigmoid") 

txt_emb_tensor = torch.as_tensor(txt_emb, device=device)
for disease in ft_probs:
    cls_idx = classes.index(disease)   
    plot_gradcam(ft_cam,sample_path,disease,cls_idx,txt_emb_tensor,ft_probs[disease])
